In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Install PySpark
!pip install -q pyspark==3.5.1 findspark pandas numpy matplotlib
print("✅ Libraries installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 14.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
✅ Libraries installed!


In [4]:
import os
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum as spark_sum
# Create Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Practical4_DecisionTree") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
print("✅ Spark session created!")
# Import ML libraries
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
print("✅ ML libraries imported!")

✅ Spark session created!
✅ ML libraries imported!


In [5]:

from google.colab import files
import zipfile
import os
uploaded = files.upload()
# Extract the ZIP file
zip_file = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("/content/cleaned_data")
print("✅ ZIP file extracted!")
# Show extracted files
for root, dirs, filenames in os.walk("/content/cleaned_data"):
    for filename in filenames:
        print(os.path.join(root, filename))

Saving retailmart_transactions_cleaned_parquet.zip to retailmart_transactions_cleaned_parquet.zip
✅ ZIP file extracted!
/content/cleaned_data/part-00000-685961ec-0e8e-4400-8cac-67aee17d56ea-c000.snappy.parquet
/content/cleaned_data/.part-00000-685961ec-0e8e-4400-8cac-67aee17d56ea-c000.snappy.parquet.crc
/content/cleaned_data/part-00001-685961ec-0e8e-4400-8cac-67aee17d56ea-c000.snappy.parquet
/content/cleaned_data/._SUCCESS.crc
/content/cleaned_data/.part-00001-685961ec-0e8e-4400-8cac-67aee17d56ea-c000.snappy.parquet.crc
/content/cleaned_data/_SUCCESS


In [6]:
df = spark.read.parquet("/content/cleaned_data")

print(f"✅ Loaded {df.count():,} rows")
df.show(5)

✅ Loaded 990,078 rows
+--------------+--------------------+-------------------+----------------+---------------+--------+----------+------------+--------------+--------------+------------+---------------+-------------+----------+-----+-------------------+
|transaction_id|         customer_id|   transaction_date|product_category|   product_name|quantity|unit_price|total_amount|store_location|payment_method|customer_age|customer_gender|loyalty_years|num_visits|churn|     transaction_ts|
+--------------+--------------------+-------------------+----------------+---------------+--------+----------+------------+--------------+--------------+------------+---------------+-------------+----------+-----+-------------------+
|  DUP-00000002|e5fd51bb-e050-43f...|2022-02-09 08:24:36|          Beauty|     Makeup Kit|     1.0|     28.85|       28.85| Las Vegas, NV|     Gift Card|        19.0|           Male|          2.0|       7.0|   No|2022-02-09 08:24:36|
|  DUP-00000003|71f4be1f-6b4d-4c8...|2025-

In [7]:
# Check the dataset structure
df.printSchema()


root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- store_location: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- customer_age: double (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- loyalty_years: double (nullable = true)
 |-- num_visits: double (nullable = true)
 |-- churn: string (nullable = true)
 |-- transaction_ts: timestamp (nullable = true)



In [8]:
# Convert churn to binary
df = df.withColumn("churn_num", when(col("churn") == "Yes", 1.0).otherwise(0.0))

# Split data
train, test = df.randomSplit([0.7, 0.3], seed=42)

print(f"Training: {train.count():,} rows")
print(f"Test: {test.count():,} rows")


Training: 693,099 rows
Test: 296,979 rows


In [9]:
# Define features
categorical_cols = ["customer_gender", "store_location", "payment_method", "product_category"]
numeric_cols = ["customer_age", "loyalty_years", "total_amount", "num_visits"]

# Build pipeline
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categorical_cols]

encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_ohe") for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=numeric_cols + [c+"_ohe" for c in categorical_cols],
    outputCol="raw_features",
    handleInvalid="keep"
)

# Scale features for SVM
scaler = StandardScaler(inputCol="raw_features", outputCol="features")

pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])

# Transform data
pipeline_model = pipeline.fit(train)
train_transformed = pipeline_model.transform(train)
test_transformed = pipeline_model.transform(test)

print("✅ Data prepared for modeling!")


✅ Data prepared for modeling!


In [11]:
print("=" * 70)
print("🌲 1. DECISION TREE (BASELINE)")
print("=" * 70)

dt = DecisionTreeClassifier(
    labelCol="churn_num",
    featuresCol="features",
    maxDepth=5,
    minInstancesPerNode=50
)

dt_model = dt.fit(train_transformed)
dt_predictions = dt_model.transform(test_transformed)

evaluator = MulticlassClassificationEvaluator(
    labelCol="churn_num",
    metricName="f1"
)

dt_f1 = evaluator.evaluate(dt_predictions)

print(f"Decision Tree F1 Score: {dt_f1:.4f}")

dt_importance = dt_model.featureImportances
print(f"Top 3 features: {dt_importance.toArray()[:3]}")


🌲 1. DECISION TREE (BASELINE)
Decision Tree F1 Score: 0.7314
Top 3 features: [0.02246227 0.83201856 0.08098036]


In [13]:

from pyspark.ml.classification import LinearSVC

print("\n" + "=" * 70)
print("📊 2. SUPPORT VECTOR MACHINE (SVM)")
print("=" * 70)

svm = LinearSVC(
    labelCol="churn_num",
    featuresCol="features",
    maxIter=100,
    regParam=0.1
)

svm_model = svm.fit(train_transformed)
svm_predictions = svm_model.transform(test_transformed)

svm_f1 = evaluator.evaluate(svm_predictions)
print(f"SVM F1 Score: {svm_f1:.4f}")



📊 2. SUPPORT VECTOR MACHINE (SVM)
SVM F1 Score: 0.6809


In [14]:
from pyspark.ml.classification import NaiveBayes

print("\n" + "=" * 70)
print("📊 3. NAIVE BAYES")
print("=" * 70)

nb = NaiveBayes(
    labelCol="churn_num",
    featuresCol="features",
    modelType="gaussian"
)

nb_model = nb.fit(train_transformed)
nb_predictions = nb_model.transform(test_transformed)

nb_f1 = evaluator.evaluate(nb_predictions)
print(f"Naive Bayes F1 Score: {nb_f1:.4f}")



📊 3. NAIVE BAYES
Naive Bayes F1 Score: 0.7113


In [15]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

print("\n" + "=" * 70)
print("📊 4. K-MEANS CLUSTERING")
print("=" * 70)

# Use only numeric features for clustering
numeric_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="cluster_features"
)

train_cluster = numeric_assembler.transform(train)

kmeans = KMeans(featuresCol="cluster_features", k=4, seed=42)
kmeans_model = kmeans.fit(train_cluster)
kmeans_predictions = kmeans_model.transform(train_cluster)

cluster_evaluator = ClusteringEvaluator(
    featuresCol="cluster_features",
    metricName="silhouette"
)

silhouette = cluster_evaluator.evaluate(kmeans_predictions)

print(f"K-Means Silhouette Score: {silhouette:.4f}")

# Display centroids
centroids = kmeans_model.clusterCenters()
print("\nCluster Centroids:")

for i, c in enumerate(centroids):
    print(f"     Cluster {i}: {c}")



📊 4. K-MEANS CLUSTERING
K-Means Silhouette Score: 0.8337

Cluster Centroids:
     Cluster 0: [41.85534174  2.43843182 69.09449834  9.88081115]
     Cluster 1: [ 41.87405614   2.43907275 341.1482448    9.87210647]
     Cluster 2: [ 41.93011984   2.45032035 929.8500966    9.92875876]
     Cluster 3: [  42.16796748    2.47707317 2117.71655447    9.93918699]


In [16]:
print("\n" + "=" * 70)
print("📊 MODEL COMPARISON")
print("=" * 70)

comparison = [
    {"Algorithm": "Decision Tree", "F1 Score": dt_f1, "Type": "Supervised"},
    {"Algorithm": "SVM", "F1 Score": svm_f1, "Type": "Supervised"},
    {"Algorithm": "Naive Bayes", "F1 Score": nb_f1, "Type": "Supervised"}
]

import pandas as pd
comparison_df = pd.DataFrame(comparison)

print("\nSupervised Model Comparison:")
print(comparison_df)

print(f"\nK-Means Silhouette Score: {silhouette:.4f}")



📊 MODEL COMPARISON

Supervised Model Comparison:
       Algorithm  F1 Score        Type
0  Decision Tree  0.731351  Supervised
1            SVM  0.680944  Supervised
2    Naive Bayes  0.711331  Supervised

K-Means Silhouette Score: 0.8337


In [17]:
print("\n" + "=" * 70)
print("🔧 ADVANCED SVM TUNING")
print("=" * 70)

# Try different regularization parameters
reg_params = [0.01, 0.1, 1.0, 10.0]
svm_results = []

for reg in reg_params:
    svm = LinearSVC(
        labelCol="churn_num",
        featuresCol="features",
        maxIter=100,
        regParam=reg
    )
    model = svm.fit(train_transformed)
    pred = model.transform(test_transformed)
    f1 = evaluator.evaluate(pred)
    svm_results.append({"regParam": reg, "F1": f1})
    print(f"regParam={reg}: F1={f1:.4f}")

svm_df = pd.DataFrame(svm_results)

print("\nSVM Tuning Results:")
print(svm_df)



🔧 ADVANCED SVM TUNING
regParam=0.01: F1=0.6809
regParam=0.1: F1=0.6809
regParam=1.0: F1=0.6809
regParam=10.0: F1=0.6809

SVM Tuning Results:
   regParam        F1
0      0.01  0.680944
1      0.10  0.680944
2      1.00  0.680944
3     10.00  0.680944


In [18]:
print("\n" + "=" * 70)
print("📊 CLUSTER PROFILING")
print("=" * 70)

# Add cluster predictions to original data
from pyspark.sql.functions import col

train_clustered = kmeans_predictions.withColumnRenamed(
    "prediction", "cluster"
)

# Profile each cluster
print("\nCluster Profiles:")

for i in range(4):
    cluster_data = train_clustered.filter(col("cluster") == i)
    count = cluster_data.count()

    # Calculate averages
    avg_age = cluster_data.selectExpr("avg(customer_age)").collect()[0][0]
    avg_loyalty = cluster_data.selectExpr("avg(loyalty_years)").collect()[0][0]
    avg_amount = cluster_data.selectExpr("avg(total_amount)").collect()[0][0]
    avg_visits = cluster_data.selectExpr("avg(num_visits)").collect()[0][0]

    # Churn rate in this cluster
    churn_rate = cluster_data.filter(col("churn") == "Yes").count() / count * 100

    print(f"\n🏷️ Cluster {i}: {count:,} transactions")
    print(f"   Avg Age: {avg_age:.1f}")
    print(f"   Avg Loyalty: {avg_loyalty:.1f} years")
    print(f"   Avg Total Amount: ${avg_amount:.2f}")
    print(f"   Avg Visits: {avg_visits:.1f}")
    print(f"   Churn Rate: {churn_rate:.1f}%")



📊 CLUSTER PROFILING

Cluster Profiles:

🏷️ Cluster 0: 535,778 transactions
   Avg Age: 41.9
   Avg Loyalty: 2.4 years
   Avg Total Amount: $69.09
   Avg Visits: 9.9
   Churn Rate: 23.0%

🏷️ Cluster 1: 121,046 transactions
   Avg Age: 41.9
   Avg Loyalty: 2.4 years
   Avg Total Amount: $341.14
   Avg Visits: 9.9
   Churn Rate: 20.4%

🏷️ Cluster 2: 30,125 transactions
   Avg Age: 41.9
   Avg Loyalty: 2.5 years
   Avg Total Amount: $929.83
   Avg Visits: 9.9
   Churn Rate: 20.1%

🏷️ Cluster 3: 6,150 transactions
   Avg Age: 42.2
   Avg Loyalty: 2.5 years
   Avg Total Amount: $2117.72
   Avg Visits: 9.9
   Churn Rate: 20.1%
